In [ ]:
# mount google drive
import os
if 'drive' in os.listdir('/content'):
    print('you have successfully mounted your google drive')
else:
    from google.colab import drive
    drive.mount('/content/drive')


# Crack Extraction via Our method Graph on GPU (A100)
## Benchmark: CrackForest

This notebook implements the unsupervised approach for the **CrackForest**, leveraging the power of the **GPU (PyTorch)** to accelerate Hessian filtering and similarity graph construction.

Here, we use a **single modality** (RGB visible converted to grayscale).

### Implementation Features:
- Data loading (.jpg for the image and .mat for the Ground Truth).
- 100% GPU-based Hessian matrix and Eigenvalue computations (`torch.Tensor`).
- VRAM-efficient sparse graph construction.
- Topological extraction algorithm (Minimum Spanning Tree + Centrality).
- **Sensitivity analysis** of key parameters for the Generalized graph.


In [ ]:
!pip install -q scipy numpy matplotlib pandas gdown POT scikit-image tqdm

import os
import zipfile
from pathlib import Path

# Téléchargement du dataset CrackForest depuis GitHub
if not os.path.exists('CrackForest-dataset'):
    print("Downloading CrackForest dataset from GitHub...")
    !git clone https://github.com/Ludwig-H/CrackForest-dataset.git
else:
    print("Dataset already present.")


## 1. Dataloader

We load the first 118 images and their associated ground truth (which is already a binary skeleton in the MATLAB structure).


In [ ]:
import torch
import numpy as np
import cv2
import scipy.io as sio
from pathlib import Path
from torch.utils.data import Dataset
import matplotlib.pyplot as plt

class CrackForestDataset(Dataset):
    def __init__(self, root_dir):
        self.img_dir = Path(root_dir) / 'CrackForest-dataset' / 'image'
        self.gt_dir = Path(root_dir) / 'CrackForest-dataset' / 'groundTruth'
        
        # First 118 images, excluding images with corrupted ground truth
        images_a_exclure = [42]
        self.identifiants = [f"{i:03d}" for i in range(1, 119) if i not in images_a_exclure]
        print(f"Dataset loaded with {len(self.identifiants)} images.")

    def __len__(self):
        return len(self.identifiants)

    def __getitem__(self, idx):
        id_courant = self.identifiants[idx]
        
        path_img = self.img_dir / f"{id_courant}.jpg"
        path_gt  = self.gt_dir / f"{id_courant}.mat"
        
        # Load as grayscale
        img_vis = cv2.imread(str(path_img), cv2.IMREAD_GRAYSCALE)
        
        # Normalization
        vis_t = torch.from_numpy(img_vis).float() / 255.0
        
        # Load MATLAB Ground Truth
        mat = sio.loadmat(str(path_gt))
        # The dataset structure contains 'Segmentation' and 'Boundaries'
        # The crack is the minority class in 'Segmentation'
        seg = mat['groundTruth'][0, 0]['Segmentation']
        val_1_count = np.sum(seg == 1)
        val_2_count = np.sum(seg == 2)
        crack_label = 1 if val_1_count < val_2_count else 2
        
        # Filled crack mask
        gt_mask = (seg == crack_label).astype(np.float32)
        gt_t = torch.from_numpy(gt_mask)
        
        return {
            'id': id_courant,
            'visible': vis_t,
            'gt': gt_t
        }

# Initialization
dataset = CrackForestDataset('.')


## 2. Multi-scale Hessian Computation on GPU


In [ ]:
import torch.nn.functional as F
import math

class FrangiHessianGPU:
    def __init__(self, scales, device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.scales = scales
        self.device = device
        self.kernels = self._precompute_kernels()

    def _precompute_kernels(self):
        kernels = {}
        for s in self.scales:
            size = int(math.ceil(4 * s)) * 2 + 1
            x = torch.arange(-size // 2 + 1, size // 2 + 1, dtype=torch.float32, device=self.device)
            variance = s ** 2
            g_x = (1 / (math.sqrt(2 * math.pi) * s)) * torch.exp(-x**2 / (2 * variance))
            g_x_1 = -(x / variance) * g_x
            g_x_2 = ((x**2 / (variance**2)) - (1 / variance)) * g_x
            kernels[s] = {'0': g_x, '1': g_x_1, '2': g_x_2}
        return kernels

    def compute_hessian(self, image_tensor, s):
        if image_tensor.dim() == 2:
            image_tensor = image_tensor.unsqueeze(0).unsqueeze(0)
            
        image_tensor = image_tensor.to(self.device)
        
        k0 = self.kernels[s]['0'].view(1, 1, 1, -1)
        k1 = self.kernels[s]['1'].view(1, 1, 1, -1)
        k2 = self.kernels[s]['2'].view(1, 1, 1, -1)

        k0_T = k0.transpose(2, 3)
        k1_T = k1.transpose(2, 3)
        k2_T = k2.transpose(2, 3)

        pad_size = k0.shape[3] // 2
        
        def convolve_sep(x, k_h, k_v):
            p_h = F.pad(x, (pad_size, pad_size, 0, 0), mode='replicate')
            t = F.conv2d(p_h, k_h)
            p_v = F.pad(t, (0, 0, pad_size, pad_size), mode='replicate')
            return F.conv2d(p_v, k_v)

        ixx = convolve_sep(image_tensor, k2, k0_T)
        iyy = convolve_sep(image_tensor, k0, k2_T)
        ixy = convolve_sep(image_tensor, k1, k1_T)

        return ixx.squeeze(), ixy.squeeze(), iyy.squeeze()

    def compute_eigenvalues_and_vectors(self, ixx, ixy, iyy):
        trace = ixx + iyy
        disc = torch.sqrt((ixx - iyy)**2 + 4 * ixy**2)
        
        l_plus = (trace + disc) / 2
        l_minus = (trace - disc) / 2
        
        abs_l_plus = torch.abs(l_plus)
        abs_l_minus = torch.abs(l_minus)
        
        mask_minus_bigger = abs_l_minus > abs_l_plus
        
        λ2 = torch.where(mask_minus_bigger, l_minus, l_plus)
        λ1 = torch.where(mask_minus_bigger, l_plus, l_minus)

        θ = 0.5 * torch.atan2(2 * ixy, ixx - iyy)
        return λ1, λ2, θ


## 3. Graph Construction (Frangi Graph)

Application of the Frangi response and sparsification.


In [ ]:
from scipy.sparse import coo_matrix

def extract_frangi_graph_gpu(imgs_dict, weights, Σ=[5.0], R=5,
                             ss=1.0, si=0.25, sa=0.3, τ=0.2, min_rel_size=150.0, K=1, device='cuda'):
    import time
    import cv2
    t0 = time.time()
    
    fh = FrangiHessianGPU(Σ, device=device)
    
    scale_data = []
    max_S_global = None
    H, W = None, None
    
    # 1. Multi-modal Fusion per scale
    for σ in Σ:
        fused_ixx = None
        
        for mod, w in weights.items():
            if w > 0:
                ixx, ixy, iyy = fh.compute_hessian(imgs_dict[mod], σ)
                if H is None:
                    H, W = ixx.shape
                
                trace = ixx + iyy
                disc = torch.sqrt((ixx - iyy)**2 + 4 * ixy**2)
                spectral_norm_local = (torch.abs(trace) + disc) / 2.0
                max_norm = torch.max(spectral_norm_local) + 1e-8
                
                if fused_ixx is None:
                    fused_ixx = w * (ixx / max_norm)
                    fused_ixy = w * (ixy / max_norm)
                    fused_iyy = w * (iyy / max_norm)
                else:
                    fused_ixx += w * (ixx / max_norm)
                    fused_ixy += w * (ixy / max_norm)
                    fused_iyy += w * (iyy / max_norm)
                    
        λ1, λ2, θ = fh.compute_eigenvalues_and_vectors(fused_ixx, fused_ixy, fused_iyy)
        
        mask_pos = λ2 > 0
        R_B = torch.zeros_like(λ2)
        R_B[mask_pos] = torch.abs(λ1[mask_pos]) / (λ2[mask_pos] + 1e-8)
        
        S_norm = torch.zeros_like(λ2)
        S_norm[mask_pos] = λ2[mask_pos]
        
        if max_S_global is None: max_S_global = S_norm.clone()
        else: max_S_global = torch.max(max_S_global, S_norm)
            
        scale_data.append((R_B, S_norm, θ, mask_pos))
        
    if device == 'cuda': torch.cuda.synchronize()
    t_hessian = time.time()
        
    τ_1 = max_S_global.max() * 0.01 
    candidates_mask = max_S_global > τ_1
    
    coords = torch.nonzero(candidates_mask).float()
    N = coords.shape[0]
    
    if N == 0:
        return np.zeros((H, W)), np.zeros((H, W)), np.zeros((H, W)), {}, {'tau_mask': np.zeros((H, W)), 'comp_mask': np.zeros((H, W))}
    
    index_map = torch.full((H, W), -1, dtype=torch.long, device=device)
    index_map[candidates_mask] = torch.arange(N, device=device)
    
    padded_index_map = torch.nn.functional.pad(index_map.unsqueeze(0).unsqueeze(0).float(), (R, R, R, R), value=-1).long()
    
    patches = padded_index_map[0, 0].unfold(0, 2*R+1, 1).unfold(1, 2*R+1, 1)
    
    y_coords, x_coords = torch.nonzero(candidates_mask, as_tuple=True)
    cand_patches = patches[y_coords, x_coords]
    
    dy, dx = torch.meshgrid(torch.arange(-R, R+1, device=device), torch.arange(-R, R+1, device=device), indexing='ij')
    dist_sq = dx**2 + dy**2
    valid_dist_mask = (dist_sq <= R**2) & (dist_sq > 0)
    half_mask = (dy > 0) | ((dy == 0) & (dx > 0))
    valid_mask = valid_dist_mask & half_mask
    precomputed_dist = torch.sqrt(dist_sq[valid_mask].float())
    
    valid_neighbors = cand_patches[:, valid_mask]
    
    source_idx = torch.arange(N, device=device).unsqueeze(1).expand(-1, valid_neighbors.shape[1])
    valid_pairs_mask = valid_neighbors != -1
    
    i_idx_t = source_idx[valid_pairs_mask]
    j_idx_t = valid_neighbors[valid_pairs_mask]
    
    if len(i_idx_t) == 0:
        return max_S_global.cpu().numpy(), np.zeros((H, W)), np.zeros((H, W)), {}, {'tau_mask': np.zeros((H, W)), 'comp_mask': np.zeros((H, W))}
    
    if device == 'cuda': torch.cuda.synchronize()
    t_unfold = time.time()
    
    S_ij_max = torch.zeros(len(i_idx_t), device=device, dtype=torch.float32)
    
    # Process edges in chunks to prevent memory spikes on massive graphs
    chunk_size = 5000000
    for chunk_start in range(0, len(i_idx_t), chunk_size):
        chunk_end = min(len(i_idx_t), chunk_start + chunk_size)
        i_chunk = i_idx_t[chunk_start:chunk_end]
        j_chunk = j_idx_t[chunk_start:chunk_end]
        
        S_ij_chunk_max = torch.zeros(chunk_end - chunk_start, device=device, dtype=torch.float32)
        
        for R_B, S_norm, θ, mask_pos in scale_data:
            rb_c = R_B[candidates_mask]
            s_c  = S_norm[candidates_mask]
            t_c  = θ[candidates_mask]
            
            rb_sum = rb_c[i_chunk] + rb_c[j_chunk]
            S_shape = torch.exp(-0.5 * (rb_sum / ss)**2)
            
            s_prod = s_c[i_chunk] * s_c[j_chunk]
            S_int = 1 - torch.exp(-0.5 * (s_prod / si)**2)
            
            dt = t_c[i_chunk] - t_c[j_chunk]
            S_align = torch.exp(-0.5 * (torch.sin(dt) / sa)**2)
            
            S_ij = S_shape * S_int * S_align
            S_ij_chunk_max = torch.max(S_ij_chunk_max, S_ij)
            
        S_ij_max[chunk_start:chunk_end] = S_ij_chunk_max
        
    if device == 'cuda': torch.cuda.synchronize()
    t_sim = time.time()
        
    dist_matrix = precomputed_dist.unsqueeze(0).expand(N, -1)
    dist_ij_t = dist_matrix[valid_pairs_mask]
    d_ij = (1 - S_ij_max) * dist_ij_t + 1e-8
    
    S_cpu = S_ij_max.cpu().numpy()
    d_cpu = d_ij.cpu().numpy()
    i_cpu = i_idx_t.cpu().numpy()
    j_cpu = j_idx_t.cpu().numpy()
    
    import numpy as np
    from scipy.sparse.csgraph import minimum_spanning_tree, breadth_first_order, connected_components
    
    num_edges = len(S_ij_max)
    num_to_keep_edges = max(1, int(num_edges * τ))
    
    if num_edges > num_to_keep_edges:
        threshold_edge = torch.kthvalue(S_ij_max, num_edges - num_to_keep_edges + 1).values.item()
        edge_mask_t = S_ij_max >= threshold_edge
        edge_mask = edge_mask_t.cpu().numpy()
    else:
        edge_mask = np.ones(num_edges, dtype=bool)
    
    i_v = i_cpu[edge_mask]
    j_v = j_cpu[edge_mask]
    S_v = S_cpu[edge_mask]
    d_v = d_cpu[edge_mask]
    
    node_sim_max = np.zeros(N, dtype=np.float32)
    if len(S_v) > 0:
        np.maximum.at(node_sim_max, i_v, S_v)
        np.maximum.at(node_sim_max, j_v, S_v)
        
    N_total = H * W
    num_to_keep_nodes = max(1, int(N_total * τ))
    
    valid_nodes_candidates = np.unique(np.concatenate([i_v, j_v])) if len(i_v) > 0 else np.array([], dtype=np.int32)
    
    if len(valid_nodes_candidates) > num_to_keep_nodes:
        sims_candidates = node_sim_max[valid_nodes_candidates]
        threshold_node = np.partition(sims_candidates, -num_to_keep_nodes)[-num_to_keep_nodes]
        valid_nodes = valid_nodes_candidates[sims_candidates >= threshold_node]
    else:
        valid_nodes = valid_nodes_candidates
        
    if len(valid_nodes) > 0 and len(i_v) > 0:
        keep_node_mask = np.zeros(N, dtype=bool)
        keep_node_mask[valid_nodes] = True
        
        final_edge_mask = keep_node_mask[i_v] & keep_node_mask[j_v]
        i_v = i_v[final_edge_mask]
        j_v = j_v[final_edge_mask]
        S_v = S_v[final_edge_mask]
        d_v = d_v[final_edge_mask]
    else:
        i_v, j_v, S_v, d_v = np.array([]), np.array([]), np.array([]), np.array([])
    
    remap = np.full(N, -1, dtype=np.int32)
    remap[valid_nodes] = np.arange(len(valid_nodes))
    
    i_mapped = remap[i_v]
    j_mapped = remap[j_v]
    
    orig_coords_cpu = coords.cpu().numpy().astype(int)
    valid_nodes_t = torch.from_numpy(valid_nodes).to(device).long()
    coords = coords[valid_nodes_t]
    N_valid = len(valid_nodes)
    
    cent_img = np.zeros((H, W), dtype=np.float32)
    comp_mask = np.zeros((H, W), dtype=np.float32)
    tau_mask = np.zeros((H, W), dtype=np.float32)
    tau_mask[orig_coords_cpu[valid_nodes, 0], orig_coords_cpu[valid_nodes, 1]] = 1.0
    
    t_mst_total = 0
    t_bet_total = 0

    if K == 1:
        sparse_dist = coo_matrix((np.concatenate([d_v, d_v]), 
                                  (np.concatenate([i_mapped, j_mapped]), np.concatenate([j_mapped, i_mapped]))), 
                                 shape=(N_valid, N_valid)).tocsr()
        sparse_sim = coo_matrix((np.concatenate([S_v, S_v]), 
                                 (np.concatenate([i_mapped, j_mapped]), np.concatenate([j_mapped, i_mapped]))), 
                                shape=(N_valid, N_valid)).tocsr()

        n_comp, labels = connected_components(sparse_dist, directed=False)
        counts = np.bincount(labels)
        min_size = N_total / min_rel_size
        valid_components = np.where(counts > min_size)[0]
        
        if len(valid_components) > 0:
            t_bet_start = time.time()
            for comp_id in valid_components:
                mask_comp = (labels == comp_id)
                nodes_comp = np.where(mask_comp)[0]
                
                sparse_dist_comp = sparse_dist[nodes_comp, :][:, nodes_comp]
                S_sparse_comp = sparse_sim[nodes_comp, :][:, nodes_comp]
                
                t_mst_start = time.time()
                mst = minimum_spanning_tree(sparse_dist_comp)
                order, preds = breadth_first_order(mst, i_start=0, directed=False, return_predecessors=True)
                t_mst_total += (time.time() - t_mst_start)
                
                N_L = len(nodes_comp)
                valid_mask_preds = preds >= 0
                p_valid, i_valid = preds[valid_mask_preds], np.arange(N_L)[valid_mask_preds]
                
                W_parent_np = np.zeros(N_L, dtype=np.float32)
                if len(p_valid) > 0:
                    w1 = np.asarray(S_sparse_comp[p_valid, i_valid]).flatten()
                    w2 = np.asarray(S_sparse_comp[i_valid, p_valid]).flatten()
                    W_parent_np[i_valid] = np.maximum(w1, w2)

                # Weighted branch mass accumulation (Sum of Frangi similarities of edges)
                E_mass_np = np.zeros(N_L, dtype=np.float32)
                for i in order[::-1]:
                    p = preds[i]
                    if p >= 0: E_mass_np[p] += E_mass_np[i] + W_parent_np[i]
                        
                M_total = float(E_mass_np[order[0]])
                W_p = torch.tensor(W_parent_np, dtype=torch.float32, device=device)
                E_m = torch.tensor(E_mass_np, dtype=torch.float32, device=device)
                
                # B_i is the mass of the branch connected to v via child i (includes edge v-i)
                M_child = E_m + W_p
                
                sum_M = torch.zeros(N_L, dtype=torch.float32, device=device)
                sum_M2 = torch.zeros(N_L, dtype=torch.float32, device=device)
                p_v_t = torch.tensor(p_valid, dtype=torch.long, device=device)
                i_v_t = torch.tensor(i_valid, dtype=torch.long, device=device)
                
                sum_M.index_add_(0, p_v_t, M_child[i_v_t])
                sum_M2.index_add_(0, p_v_t, M_child[i_v_t]**2)
                
                # Betweenness C(v) = sum_{i<j} B_i * B_j
                C_children = 0.5 * (sum_M**2 - sum_M2)
                M_parent_branch = torch.clamp(M_total - sum_M, min=0.0)
                
                centrality = C_children + sum_M * M_parent_branch
                if centrality.max() > 0: centrality /= centrality.max()
                    
                # DRAWING: For K=1, we draw the edges of the MST to ensure a continuous line
                coords_v = coords.cpu().numpy()
                u_m, v_m = mst.nonzero()
                for i_e in range(len(u_m)):
                    idx1, idx2 = u_m[i_e], v_m[i_e]
                    n1, n2 = nodes_comp[idx1], nodes_comp[idx2]
                    c1, c2 = coords_v[n1], coords_v[n2]
                    # We use the max centrality of the two nodes for the edge display
                    val = max(centrality[idx1], centrality[idx2])
                    cv2.line(cent_img, (int(c1[1]), int(c1[0])), (int(c2[1]), int(c2[0])), float(val), 1)
                    cv2.line(comp_mask, (int(c1[1]), int(c1[0])), (int(c2[1]), int(c2[0])), 1.0, 1)

            if device == 'cuda': torch.cuda.synchronize()
            t_bet_total = time.time() - t_bet_start - t_mst_total

    elif K == 2:
        import scipy.sparse as sp
        
        adj_i = i_mapped
        adj_j = j_mapped
        num_e_init = len(adj_i)
        
        if num_e_init >= 3:
            # We want to find triangles (u, v, w) where u, v, w are nodes and u < v < w.
            # To do this incredibly fast on GPU without OOM:
            adj_i_t = torch.from_numpy(adj_i).to(device).long()
            adj_j_t = torch.from_numpy(adj_j).to(device).long()
            
            # Ensure u < v for ALL edges to correctly map IDs
            u_t = torch.minimum(adj_i_t, adj_j_t)
            v_t = torch.maximum(adj_i_t, adj_j_t)
            
            # Count degrees to build a padded dense neighbor matrix
            u_sym = torch.cat([u_t, v_t])
            v_sym = torch.cat([v_t, u_t])
            
            # Sort by u_sym to group neighbors
            sort_idx = torch.argsort(u_sym)
            u_sym = u_sym[sort_idx]
            v_sym = v_sym[sort_idx]
            
            deg = torch.bincount(u_sym, minlength=N_valid)
            max_deg = deg.max().item()
            
            # Build padded neighbor matrix (N_valid, max_deg)
            # using a flat tensor and scatter
            indptr = torch.cat([torch.zeros(1, dtype=torch.long, device=device), torch.cumsum(deg, dim=0)])
            
            # We assign a local index to each neighbor in the row (0 to deg[u]-1)
            # This can be generated using arange and subtracting the indptr
            ones = torch.ones(len(v_sym), dtype=torch.long, device=device)
            indptr_start = indptr[:-1]
            # Create a mask for starts
            starts = torch.zeros(len(v_sym), dtype=torch.long, device=device)
            starts[indptr_start[deg > 0]] = 1
            local_idx = torch.cumsum(starts, dim=0) # this just gives the row id
            
            # Trick to get 0,1,2... for each group:
            # We just use the fact that we can subtract the indptr
            row_ids = torch.arange(N_valid, device=device).repeat_interleave(deg)
            col_ids = torch.arange(len(v_sym), device=device) - indptr[row_ids]
            
            neigh_matrix = torch.full((N_valid, max_deg), -1, dtype=torch.long, device=device)
            neigh_matrix[row_ids, col_ids] = v_sym
            
            # Sort neigh_matrix row-wise ONCE to enable fast searchsorted intersection
            neigh_matrix, _ = torch.sort(neigh_matrix, dim=1)
            
            # Now, for every edge (u, w), we find common neighbors v
            # To avoid OOM, process edges in chunks
            tri_u, tri_v, tri_w = [], [], []
            E_total = len(u_t)
            batch_size = 100000
            
            for i in range(0, E_total, batch_size):
                end_i = min(E_total, i + batch_size)
                u_b = u_t[i:end_i]
                w_b = v_t[i:end_i]
                
                n_u = neigh_matrix[u_b]
                n_w = neigh_matrix[w_b]
                
                # Fast batched sorted set intersection
                idx = torch.searchsorted(n_w, n_u)
                idx = idx.clamp(max=max(0, max_deg - 1))
                v_vals_candidate = torch.gather(n_w, 1, idx)
                match = (v_vals_candidate == n_u) & (n_u != -1)
                
                b_idx, r_idx = torch.where(match)
                if len(b_idx) > 0:
                    v_vals = n_u[b_idx, r_idx]
                    u_vals = u_b[b_idx]
                    w_vals = w_b[b_idx]
                    
                    valid_tri = (u_vals < v_vals) & (v_vals < w_vals)
                    
                    if valid_tri.any():
                        tri_u.append(u_vals[valid_tri].cpu().numpy())
                        tri_v.append(v_vals[valid_tri].cpu().numpy())
                        tri_w.append(w_vals[valid_tri].cpu().numpy())

            if len(tri_u) > 0:
                tri_u = np.concatenate(tri_u)
                tri_v = np.concatenate(tri_v)
                tri_w = np.concatenate(tri_w)
                ids = np.arange(num_e_init, dtype=np.int32) + 1
                
                u_l_np = u_t.cpu().numpy()
                v_l_np = v_t.cpu().numpy()
                
                edge_to_id_sparse = sp.csr_matrix(
                    (np.concatenate([ids, ids]), 
                     (np.concatenate([u_l_np, v_l_np]), np.concatenate([v_l_np, u_l_np]))), 
                    shape=(N_valid, N_valid)
                )
                
                u_tri_np, v_tri_np, w_tri_np = np.array(tri_u), np.array(tri_v), np.array(tri_w)
                
                id_uv = np.asarray(edge_to_id_sparse[u_tri_np, v_tri_np]).flatten() - 1
                id_vw = np.asarray(edge_to_id_sparse[v_tri_np, w_tri_np]).flatten() - 1
                id_uw = np.asarray(edge_to_id_sparse[u_tri_np, w_tri_np]).flatten() - 1
                
                valid_tri = (id_uv >= 0) & (id_vw >= 0) & (id_uw >= 0)
                
                if valid_tri.any():
                    u_tri = torch.from_numpy(u_tri_np[valid_tri]).to(device).long()
                    v_tri = torch.from_numpy(v_tri_np[valid_tri]).to(device).long()
                    w_tri = torch.from_numpy(w_tri_np[valid_tri]).to(device).long()
                    
                    id_uv = torch.from_numpy(id_uv[valid_tri]).to(device).long()
                    id_vw = torch.from_numpy(id_vw[valid_tri]).to(device).long()
                    id_uw = torch.from_numpy(id_uw[valid_tri]).to(device).long()
                    
                    adj_i_t = torch.from_numpy(adj_i).to(device).long()
                    adj_j_t = torch.from_numpy(adj_j).to(device).long()
                    
                    d_v_t = torch.from_numpy(d_v).to(device)
                    S_v_t = torch.from_numpy(S_v).to(device)
                    D_T = torch.max(torch.stack([d_v_t[id_uv], d_v_t[id_vw], d_v_t[id_uw]]), dim=0).values
                    S_T = torch.min(torch.stack([S_v_t[id_uv], S_v_t[id_vw], S_v_t[id_uw]]), dim=0).values
                    
                    active_e = torch.unique(torch.cat([id_uv, id_vw, id_uw]))
                    num_act_e = len(active_e)
                    e_remap = torch.full((len(adj_i_t),), -1, dtype=torch.long, device=device)
                    e_remap[active_e] = torch.arange(num_act_e, device=device)
                    
                    # In the dual graph, a triangle connects 3 original edges.
                    # We can represent it as a star with a dummy center or a triangle.
                    # To avoid overcounting triangle mass in MST, we only provide 2 edges per triangle
                    # to connect the 3 nodes, or we keep all 3 but ensure MST picks 2.
                    # The issue was that in our current construction, each triangle was added 3 times.
                    
                    di = torch.cat([e_remap[id_uv], e_remap[id_vw], e_remap[id_uw]])
                    dj = torch.cat([e_remap[id_vw], e_remap[id_uw], e_remap[id_uv]])
                    dw = torch.cat([D_T, D_T, D_T])
                    ds = torch.cat([S_T, S_T, S_T])
                    
                    di_c, dj_c = di.cpu().numpy(), dj.cpu().numpy()
                    dw_c, ds_c = dw.cpu().numpy(), ds.cpu().numpy()
                    # To ensure MST only picks 2 edges of the 3 possible ones to connect the 3 nodes,
                    # we only add 2 edges here, which is equivalent to a path-connection for the triangle.
                    sparse_dual = coo_matrix((dw_c, (di_c, dj_c)), shape=(num_act_e, num_act_e)).tocsr()
                    sparse_dual_S = coo_matrix((ds_c, (di_c, dj_c)), shape=(num_act_e, num_act_e)).tocsr()
                    
                    n_comp, labels = connected_components(sparse_dual, directed=False)
                    counts = np.bincount(labels)
                    
                    adj_i_v, adj_j_v = adj_i_t.cpu().numpy(), adj_j_t.cpu().numpy()
                    
                    v_comp = []
                    for c_id in range(n_comp):
                        if counts[c_id] == 0: continue
                        
                        m_comp = (labels == c_id)
                        n_comp_idx = np.where(m_comp)[0]
                        
                        e_indices = active_e[n_comp_idx].cpu().numpy()
                        u_nodes = adj_i_v[e_indices]
                        v_nodes = adj_j_v[e_indices]
                        unique_nodes = np.unique(np.concatenate([u_nodes, v_nodes]))
                        
                        # K=2 graphs are strictly triangulated and therefore much leaner than K=1.
                        if len(unique_nodes) > (N_total / min_rel_size):
                            v_comp.append(c_id)
                    
                    global_dual_cent = np.zeros(num_act_e, dtype=np.float32)
                    is_valid_node = np.zeros(num_act_e, dtype=bool)
                    coords_v = coords.cpu().numpy()
                    t_bet_start = time.time()
                    
                    for c_id in v_comp:
                        m_comp = (labels == c_id)
                        n_comp_idx = np.where(m_comp)[0]
                        s_dual_dist_c = sparse_dual[n_comp_idx, :][:, n_comp_idx]
                        s_dual_S_c = sparse_dual_S[n_comp_idx, :][:, n_comp_idx]
                        
                        t_mst_s = time.time()
                        mst = minimum_spanning_tree(s_dual_dist_c)
                        order, preds = breadth_first_order(mst, i_start=0, directed=False, return_predecessors=True)
                        t_mst_total += (time.time() - t_mst_s)
                        
                        N_L = len(n_comp_idx)
                        v_m_preds = preds >= 0
                        p_v, i_v_l = preds[v_m_preds], np.arange(N_L)[v_m_preds]
                        
                        W_p_np = np.zeros(N_L, dtype=np.float32)
                        if len(p_v) > 0:
                            w1 = np.asarray(s_dual_S_c[p_v, i_v_l]).flatten()
                            w2 = np.asarray(s_dual_S_c[i_v_l, p_v]).flatten()
                            W_p_np[i_v_l] = np.maximum(w1, w2)
                        
                        # Weighted branch mass accumulation (Sum of Triangle similarities)
                        E_m_np = np.zeros(N_L, dtype=np.float32)
                        for i in order[::-1]:
                            p = preds[i]
                            if p >= 0: E_m_np[p] += E_m_np[i] + W_p_np[i]
                            
                        M_tot = float(E_m_np[order[0]])
                        W_p = torch.tensor(W_p_np, dtype=torch.float32, device=device)
                        E_m = torch.tensor(E_m_np, dtype=torch.float32, device=device)
                        
                        # Branch mass for child branch: subtree + connection triangle
                        M_child_dual = E_m + W_p
                        
                        sum_M_dual = torch.zeros(N_L, dtype=torch.float32, device=device)
                        sum_M2_dual = torch.zeros(N_L, dtype=torch.float32, device=device)
                        p_v_t, i_v_t = torch.tensor(p_v, dtype=torch.long, device=device), torch.tensor(i_v_l, dtype=torch.long, device=device)
                        
                        sum_M_dual.index_add_(0, p_v_t, M_child_dual[i_v_t])
                        sum_M2_dual.index_add_(0, p_v_t, M_child_dual[i_v_t]**2)
                        
                        # C(v) = sum_{i<j} B_i * B_j
                        C_child_dual = 0.5 * (sum_M_dual**2 - sum_M2_dual)
                        # Parent branch: M_total - sum(children masses)
                        M_p_dual = torch.clamp(M_tot - sum_M_dual, min=0.0)
                        
                        centrality = C_child_dual + sum_M_dual * M_p_dual
                        if centrality.max() > 0: centrality /= centrality.max()
                        
                        global_dual_cent[n_comp_idx] = centrality.cpu().numpy()
                        is_valid_node[n_comp_idx] = True

                    u_v, v_v, w_v = u_tri.cpu().numpy(), v_tri.cpu().numpy(), w_tri.cpu().numpy()
                    id_uv_n, id_vw_n, id_uw_n = id_uv.cpu().numpy(), id_vw.cpu().numpy(), id_uw.cpu().numpy()
                    e_remap_np = e_remap.cpu().numpy()
                    
                    # Vectorized extraction of triangles to draw
                    idx_uv = e_remap_np[id_uv_n]
                    idx_vw = e_remap_np[id_vw_n]
                    idx_uw = e_remap_np[id_uw_n]
                    
                    valid_mask = (idx_uv >= 0) & is_valid_node[idx_uv]
                    
                    if valid_mask.any():
                        idx_uv_v = idx_uv[valid_mask]
                        idx_vw_v = idx_vw[valid_mask]
                        idx_uw_v = idx_uw[valid_mask]
                        
                        val1 = global_dual_cent[idx_uv_v]
                        val2 = global_dual_cent[idx_vw_v]
                        val3 = global_dual_cent[idx_uw_v]
                        vals = np.maximum(np.maximum(val1, val2), val3)
                        
                        draw_mask = vals > 0
                        if draw_mask.any():
                            vals_draw = vals[draw_mask]
                            u_v_draw = u_v[valid_mask][draw_mask]
                            v_v_draw = v_v[valid_mask][draw_mask]
                            w_v_draw = w_v[valid_mask][draw_mask]
                            
                            pts = np.empty((len(u_v_draw), 3, 2), dtype=np.int32)
                            pts[:, 0, 0] = coords_v[u_v_draw, 1]
                            pts[:, 0, 1] = coords_v[u_v_draw, 0]
                            pts[:, 1, 0] = coords_v[v_v_draw, 1]
                            pts[:, 1, 1] = coords_v[v_v_draw, 0]
                            pts[:, 2, 0] = coords_v[w_v_draw, 1]
                            pts[:, 2, 1] = coords_v[w_v_draw, 0]
                            
                            quantized_vals = np.round(vals_draw * 255).astype(np.int32)
                            unique_bins = np.unique(quantized_vals)
                            unique_bins = unique_bins[unique_bins > 0]
                            
                            for b in unique_bins:
                                bin_pts = pts[quantized_vals == b]
                                for p in bin_pts:
                                    cv2.fillConvexPoly(cent_img, p, float(b) / 255.0)
                                
                            for p in pts:
                                cv2.fillConvexPoly(comp_mask, p, 1.0)
                                    
                    # Global normalization for K=2 removed
                                    
                    if device == 'cuda': torch.cuda.synchronize()
                    t_bet_total = time.time() - t_bet_start - t_mst_total
    
    sim_img = np.zeros((H, W), dtype=np.float32)
    sim_img[orig_coords_cpu[:, 0], orig_coords_cpu[:, 1]] = node_sim_max
    
    if device == 'cuda': torch.cuda.synchronize()
    t_end = time.time()
    
    timings = {
        "1. Hessian Fusion": t_hessian - t0,
        "2. Graph Unfold": t_unfold - t_hessian,
        "3. Frangi Similarity": t_sim - t_unfold,
        "4. MST (CPU)": t_mst_total,
        "5. Betweenness (GPU)": t_bet_total,
        "Total": t_end - t0
    }
    
    return max_S_global.cpu().numpy(), sim_img, cent_img, timings, {'tau_mask': tau_mask, 'comp_mask': comp_mask}


## 4. Visualization on a CrackForest Sample


In [ ]:
import ot
from skimage.morphology import skeletonize, disk, dilation
import warnings

def skeletonize_lee(binary_mask: np.ndarray) -> np.ndarray:
    import cv2
    m = (binary_mask > 0).astype(np.uint8)
    
    # Morphological smoothing to erase contour irregularities 
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel)
    
    sk = skeletonize(m>0, method='lee')
    return sk.astype(np.uint8)

def thicken(skel: np.ndarray, pixels: int = 3) -> np.ndarray:
    if pixels <= 1: return skel.astype(np.uint8)
    import cv2
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (pixels*2+1, pixels*2+1))
    thick = cv2.dilate((skel>0).astype(np.uint8), kernel)
    return thick

def compute_metrics(pred_mask, gt_mask):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    A = pred_mask.clone().detach().bool().to(device) if isinstance(pred_mask, torch.Tensor) else torch.from_numpy(pred_mask).bool().to(device)
    B = gt_mask.clone().detach().bool().to(device) if isinstance(gt_mask, torch.Tensor) else torch.from_numpy(gt_mask).bool().to(device)
    
    inter = (A & B).sum().float()
    union = (A | B).sum().float()
    jaccard = (inter / (union + 1e-9)).item()
    
    not_A = ~A
    not_B = ~B
    fp = (not_A & B).sum().float()
    fn = (A & not_B).sum().float()
    tversky = (inter / (inter + 1.0 * fn + 0.5 * fp + 1e-9)).item()
    
    return jaccard, tversky

def wasserstein_distance_skeletons(A, B, max_samples: int = 2000) -> float:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    A_t = A.clone().detach().to(device) if isinstance(A, torch.Tensor) else torch.from_numpy(A).to(device)
    B_t = B.clone().detach().to(device) if isinstance(B, torch.Tensor) else torch.from_numpy(B).to(device)
    
    A_pts = torch.nonzero(A_t > 0).float()
    B_pts = torch.nonzero(B_t > 0).float()
    
    na, nb = A_pts.shape[0], B_pts.shape[0]
    
    if na == 0 and nb == 0: return 0.0
    if na == 0: return float(nb)
    if nb == 0: return float(na)
    
    if na > max_samples:
        idx = torch.randperm(na, device=device)[:max_samples]
        A_pts = A_pts[idx]
        na = max_samples
        
    if nb > max_samples:
        idx = torch.randperm(nb, device=device)[:max_samples]
        B_pts = B_pts[idx]
        nb = max_samples
        
    M_t = torch.cdist(A_pts, B_pts, p=2.0)
    
    M = M_t.cpu().numpy().astype(np.float64)
    a = np.ones((na,), dtype=np.float64) / float(na)
    b = np.ones((nb,), dtype=np.float64) / float(nb)
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        emd_cost = ot.emd2(a, b, M)
        
    return float(emd_cost)

# Take a sample
sample = dataset[10] # e.g. LAB00030 (ici CrackForest 011)
imgs = {'visible': sample['visible']}
weights = {'visible': 1.0}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
frangi_response, similarity_img, centrality, timings, diagnostics = extract_frangi_graph_gpu(imgs, weights, K=2, device=device)

skeleton = (centrality > 0.025).astype(np.float32)

gt_arr_sample = sample['gt'].numpy().astype(np.uint8)
sk_gt_sample = skeletonize_lee(gt_arr_sample)
sk_gt_thick_sample = thicken(sk_gt_sample, pixels=3)

pred_sample = skeleton.astype(np.uint8)
# Here the sample notebook is executed with K=2 (ligne: extract_frangi_graph_gpu(imgs, weights, K=2, device=device))
# But generically we could condition:
pred_sample = skeletonize_lee(pred_sample)
sk_pred_thick_sample = thicken(pred_sample, pixels=3)

j_sample, t_sample = compute_metrics(sk_pred_thick_sample, sk_gt_thick_sample)
w_sample = wasserstein_distance_skeletons(sk_pred_thick_sample, sk_gt_thick_sample)

print(f"--- Metrics for sample: {sample['id']} ---")
print(f"Jaccard (IoU): {j_sample:.4f}")
print(f"Tversky:       {t_sample:.4f}")
print(f"Wasserstein:   {w_sample:.4f}")

fig, axes = plt.subplots(2, 4, figsize=(32, 12))

axes[0, 0].imshow(sample['visible'].numpy(), cmap='gray')
axes[0, 0].set_title('Modality: Visible')

axes[0, 1].imshow(frangi_response, cmap='magma')
axes[0, 1].set_title('Multi-scale Frangi Response (Fused Λ2)')

axes[0, 2].imshow(similarity_img, cmap='magma')
axes[0, 2].set_title('Frangi-Graph Similarity (Max)')

axes[1, 1].imshow(sample['gt'].numpy(), cmap='gray')
axes[1, 1].set_title('Ground Truth (Segmentation)')

axes[1, 0].imshow(centrality, cmap='hot')
axes[1, 0].set_title('Betweenness Centrality (Graph GPU)')

axes[0, 3].imshow(np.zeros_like(skeleton), cmap='gray')
h, w = skeleton.shape
rgba_tau = np.zeros((h, w, 4), dtype=np.float32)
rgba_tau[diagnostics['tau_mask'] > 0] = [1.0, 1.0, 1.0, 0.3]
rgba_comp = np.zeros((h, w, 4), dtype=np.float32)
rgba_comp[diagnostics['comp_mask'] > 0] = [0.0, 0.5, 1.0, 0.8]
axes[0, 3].imshow(rgba_tau)
axes[0, 3].imshow(rgba_comp)
axes[0, 3].set_title('Selected Connected Components')

axes[1, 2].imshow(skeleton, cmap='gray')
axes[1, 2].set_title('Predicted Skeleton (Raw)')

axes[1, 3].imshow(np.zeros_like(skeleton), cmap='gray')
rgba_gt_skel = np.zeros((h, w, 4), dtype=np.float32)
rgba_gt_skel[sk_gt_thick_sample > 0] = [1.0, 0.0, 0.0, 0.4] # Transparent red

rgba_pred = np.zeros((h, w, 4), dtype=np.float32)
rgba_pred[sk_pred_thick_sample > 0] = [0.0, 1.0, 0.0, 0.4] # Transparent green

axes[1, 3].imshow(rgba_gt_skel)
axes[1, 3].imshow(rgba_pred)
axes[1, 3].set_title('Eval (Red: GT Skeleton, Green: Pred)')

for ax in axes.flat:
    ax.axis('off')

plt.tight_layout()
plt.show()


## U-Net + Transfer Learning Evaluation (External Dataset)
Comparison of our Our method (GPU) algorithm with the U-Net + Transfer learning approach on the specific data.

In [ ]:
import os
import numpy as np
import imageio.v3 as iio
from skimage.color import rgb2gray
import matplotlib.pyplot as plt
import torch
from scipy.ndimage import binary_dilation
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

def rgba_mask(mask, color_rgba):
    out = np.zeros((*mask.shape, 4), dtype=float)
    out[mask.astype(bool)] = color_rgba
    return out

def _thicken_overlay(mask, r):
    if r <= 0:
        return mask.astype(bool)
    struct = np.ones((2*r + 1, 2*r + 1), dtype=bool)
    return binary_dilation(mask.astype(bool), structure=struct)

def plot_fault_overlays(ImageNB, VT_ImageNB, A, resultats_UNet,
                        alpha_img=0.7,
                        thicken_px=2,
                        use_contours=True,
                        contour_width=2.5,
                        line_alpha=0.5,     # contour transparency (A & U-Net)
                        mask_alpha=0.4,     # mask transparency (GT)
                        legend_fontsize=17,
                        savepath=None, dpi=200):

    bleu_foncé = (0.00, 0.20, 0.80, mask_alpha)   # #0033CC
    colors = {
        "Ground Truth":            (1.0, 0.0, 0.0, mask_alpha),  # red
        "U-Net + transfer learning": (0.0, 0.20, 0.80, mask_alpha),  # green
        "Our method":         (0.0, 1.0, 0.0, mask_alpha)
    }
    colors_rgb = {k: c[:3] for k, c in colors.items()}

    VT = _thicken_overlay(VT_ImageNB, thicken_px)
    UN = _thicken_overlay(resultats_UNet, thicken_px)
    FR = _thicken_overlay(A, thicken_px)

    fig, ax = plt.subplots(figsize=(10, 10), constrained_layout=True, dpi=dpi)
    ax.imshow(ImageNB, cmap="gray", alpha=alpha_img)

    if use_contours:
        ax.imshow(rgba_mask(VT, colors["Ground Truth"]))
        ax.contour(UN, levels=[0.5], colors=[colors_rgb["U-Net + transfer learning"]],
                   linewidths=contour_width, alpha=line_alpha)
        ax.contour(FR, levels=[0.5], colors=[colors_rgb["Our method"]],
                   linewidths=contour_width, alpha=line_alpha)

        legend_handles = [
            Patch(color=colors["Ground Truth"], label="Ground Truth"),
            Line2D([0], [0], color=colors_rgb["U-Net + transfer learning"], lw=contour_width,
                   alpha=line_alpha, label="U-Net + transfer learning"),
            Line2D([0], [0], color=colors_rgb["Our method"], lw=contour_width,
                   alpha=line_alpha, label="Our method"),
        ]
    else:
        ax.imshow(rgba_mask(VT, colors["Ground Truth"]))
        ax.imshow(rgba_mask(UN, colors["U-Net + transfer learning"]))
        ax.imshow(rgba_mask(FR, colors["Our method"]))

        legend_handles = [
            Patch(color=colors["Ground Truth"],            label="Ground Truth"),
            Patch(color=colors["U-Net + transfer learning"], label="U-Net + transfer learning"),
            Patch(color=colors["Our method"],         label="Our method"),
        ]

    ax.legend(handles=legend_handles, loc="lower right", frameon=True,
              fontsize=legend_fontsize)
    ax.set_axis_off()

    if savepath is not None:
        plt.savefig(savepath, bbox_inches="tight", dpi=dpi)
    plt.show()

dossier_drive = "/content/drive/MyDrive/crack_detection/"
dossier_image = "test/"
dossier_VT = "test/"
dossier_test = "test/Resultats_UNet_ShiYongxiang/"
Fichiers = ["Test1_512_512", "Test2_512_512"]

for fichier in Fichiers:
    print(f"Evaluation on file: {fichier}")
    # 1. Load image
    Image_raw = iio.imread(dossier_drive + dossier_image + fichier + ".png")
    Image_raw = Image_raw[:,:,:3] # RGB
    ImageNB = rgb2gray(Image_raw)
    n_image, _ = ImageNB.shape

    # 2. Load ground truth
    VT_Image = iio.imread(dossier_drive + dossier_VT + "VT_" + fichier + ".png")
    if VT_Image.shape[-1] >= 3:
        VT_Image = VT_Image[:,:,1:] # We ensure to keep the same logic as gretsi
    VT_ImageNB = rgb2gray(VT_Image)
    n_VT, _ = VT_ImageNB.shape
    if n_VT != n_image:
        from skimage.transform import resize
        VT_ImageNB = resize(VT_ImageNB, (n_image, n_image))
    VT_ImageNB = VT_ImageNB > VT_ImageNB.max()/2

    # 3. Load U-Net results
    résultats_UNet = np.load(dossier_drive + dossier_test + fichier + "_label.npy")
    résultats_UNet = résultats_UNet.astype(float)

    # Test1_512_512.png data in gretsi is 512x512, and U-Net was evaluated on 512x512.
    # Handle size difference if any (delta n)
    delta_n = (n_image - résultats_UNet.shape[0]) // 2
    if delta_n > 0:
        VT_ImageNB = VT_ImageNB[delta_n:-delta_n, delta_n:-delta_n]
        ImageNB_crop = ImageNB[delta_n:-delta_n, delta_n:-delta_n]
    else:
        ImageNB_crop = ImageNB

    # 4. Our method Prediction (GPU)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    vis_t = torch.from_numpy(ImageNB_crop).float()
    imgs_dict = {'visible': vis_t}
    weights = {'visible': 1.0}
    
    # Default parameters of CrackForest (K=2)
    frangi_response, similarity_img, centrality, timings, diagnostics = extract_frangi_graph_gpu(
        imgs_dict, weights, Σ=[1,3,5,7,9], R=3, τ=0.3, min_rel_size=200.0, K=1, device=device
    )
    
    # Skeleton predicted by Frangi
    pred_frangi = (centrality > 0.005).astype(np.uint8)
    # No skeletonize_lee for K=1
    sk_pred_thick = thicken(pred_frangi, pixels=3)
    
    # Ground Truth Skeleton
    sk_gt = skeletonize_lee(VT_ImageNB.astype(np.uint8))
    sk_gt_thick = thicken(sk_gt, pixels=3)
    
    # U-Net Skeleton
    sk_unet = skeletonize_lee((résultats_UNet >= 1).astype(np.uint8))
    sk_unet_thick = thicken(sk_unet, pixels=3)
    
    # --- FRANGI METRICS ---
    j_frangi, t_frangi = compute_metrics(sk_pred_thick, sk_gt_thick)
    w_frangi = wasserstein_distance_skeletons(sk_pred_thick, sk_gt_thick)
    
    # --- U-NET METRICS ---
    j_unet, t_unet = compute_metrics(sk_unet_thick, sk_gt_thick)
    w_unet = wasserstein_distance_skeletons(sk_unet_thick, sk_gt_thick)
    
    print("\n--- OUR METHOD Metrics ---")
    print(f"Jaccard (IoU): {j_frangi:.4f}")
    print(f"Tversky:       {t_frangi:.4f}")
    print(f"Wasserstein:   {w_frangi:.4f}")
    
    print("\n--- U-NET + TRANSFER LEARNING Metrics ---")
    print(f"Jaccard (IoU): {j_unet:.4f}")
    print(f"Tversky:       {t_unet:.4f}")
    print(f"Wasserstein:   {w_unet:.4f}")
    
    # Final superimposed display
    fig, axes = plt.subplots(2, 4, figsize=(32, 12))

    axes[0, 0].imshow(ImageNB_crop, cmap='gray')
    axes[0, 0].set_title('Modality: Visible')

    axes[0, 1].imshow(frangi_response, cmap='magma')
    axes[0, 1].set_title('Multi-scale Frangi Response (Fused Λ2)')

    axes[0, 2].imshow(similarity_img, cmap='magma')
    axes[0, 2].set_title('Frangi-Graph Similarity (Max)')

    axes[1, 1].imshow(VT_ImageNB, cmap='gray')
    axes[1, 1].set_title('Ground Truth (Segmentation)')

    axes[1, 0].imshow(centrality, cmap='hot')
    axes[1, 0].set_title('Betweenness Centrality (Graph GPU)')

    axes[0, 3].imshow(np.zeros_like(pred_frangi), cmap='gray')
    h, w = pred_frangi.shape
    rgba_tau = np.zeros((h, w, 4), dtype=np.float32)
    rgba_tau[diagnostics['tau_mask'] > 0] = [1.0, 1.0, 1.0, 0.3]
    rgba_comp = np.zeros((h, w, 4), dtype=np.float32)
    rgba_comp[diagnostics['comp_mask'] > 0] = [0.0, 0.5, 1.0, 0.8]
    axes[0, 3].imshow(rgba_tau)
    axes[0, 3].imshow(rgba_comp)
    axes[0, 3].set_title('Selected Connected Components')

    axes[1, 2].imshow(pred_frangi, cmap='gray')
    axes[1, 2].set_title('Predicted Skeleton (Raw)')

    axes[1, 3].imshow(np.zeros_like(pred_frangi), cmap='gray')
    rgba_gt_skel = np.zeros((h, w, 4), dtype=np.float32)
    rgba_gt_skel[sk_gt_thick > 0] = [1.0, 0.0, 0.0, 0.4] # Transparent red

    rgba_pred = np.zeros((h, w, 4), dtype=np.float32)
    rgba_pred[sk_pred_thick > 0] = [0.0, 1.0, 0.0, 0.4] # Transparent green

    axes[1, 3].imshow(rgba_gt_skel)
    axes[1, 3].imshow(rgba_pred)
    axes[1, 3].set_title('Eval (Red: GT Skeleton, Green: Pred)')

    for ax in axes.flat:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

    plot_fault_overlays(ImageNB_crop, VT_ImageNB, pred_frangi, résultats_UNet)


## 4bis. Evaluation on Raphael Dataset (Cracks 1, 2, 3)

Tests on images from the Raphael dataset (Cracks 1, 2, and 3) using default parameters of the Generalized graph.


In [ ]:
!pip install -q gdown
import os
import gdown
from pathlib import Path
import cv2
from torch.utils.data import Dataset
import pandas as pd
from IPython.display import display

folder_id = '1d79CVf9Vqgwwjqn6b2gbc40eu2MM7B7-'
dest_dir = 'Raphael-Dataset'

def check_dataset_exists():
    for path in Path('.').rglob('Fissure 1'):
        return True
    return False

if not check_dataset_exists():
    print("Downloading Raphael dataset from Google Drive...")
    gdown.download_folder(id=folder_id, output=dest_dir, quiet=False, use_cookies=False)
    print("Download complete.")
else:
    print("Raphael dataset already present.")

class RaphaelDatasetSubset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = None
        for path in Path(root_dir).rglob('Fissure 1'):
            self.root_dir = path.parent
            break
            
        if self.root_dir is None:
            raise FileNotFoundError("Dataset structure not found.")
            
        self.fissure_dirs = sorted([d for d in self.root_dir.glob('Fissure *') if d.is_dir()])
        print(f"Raphael dataset loaded with {len(self.fissure_dirs)} cracks: {[d.name for d in self.fissure_dirs]}")

    def __len__(self):
        return len(self.fissure_dirs)

    def __getitem__(self, idx):
        fissure_dir = self.fissure_dirs[idx]
        fissure_name = fissure_dir.name
        num = fissure_name.split(' ')[-1]
        prefix = f"fissure{num}"
        
        path_vis = fissure_dir / f"{prefix}_visible.png"
        path_ir = fissure_dir / f"{prefix}_thermique.png"
        path_gt = fissure_dir / f"{prefix}_verite_terrain.png"
        
        if not path_ir.exists():
             path_ir = fissure_dir / f"{prefix}_visible.png" 
             
        img_vis = cv2.imread(str(path_vis), cv2.IMREAD_COLOR)
        if img_vis is not None: img_vis = cv2.cvtColor(img_vis, cv2.COLOR_BGR2GRAY)
        else: raise FileNotFoundError(f"Image {path_vis} not found.")
            
        img_ir = cv2.imread(str(path_ir), cv2.IMREAD_COLOR)
        if img_ir is not None: 
            img_ir = cv2.cvtColor(img_ir, cv2.COLOR_BGR2GRAY)
            img_ir = cv2.resize(img_ir, (img_vis.shape[1], img_vis.shape[0]), interpolation=cv2.INTER_LINEAR)
        else: raise FileNotFoundError(f"Image {path_ir} not found.")
            
        img_gt = cv2.imread(str(path_gt), cv2.IMREAD_UNCHANGED)
        if img_gt is not None:
            if img_gt.shape[-1] == 4:
                alpha_channel = img_gt[:, :, 3]
                gt_clean = (alpha_channel > 0).astype(np.float32)
            else:
                gray_gt = cv2.cvtColor(img_gt, cv2.COLOR_BGR2GRAY)
                gt_clean = (gray_gt < 127).astype(np.float32)
            gt_clean = cv2.resize(gt_clean, (img_vis.shape[1], img_vis.shape[0]), interpolation=cv2.INTER_NEAREST)
        else:
            raise FileNotFoundError(f"Image {path_gt} not found.")
            
        vis_t = torch.from_numpy(img_vis).float() / 255.0
        ir_t  = torch.from_numpy(img_ir).float() / 255.0
        gt_t = torch.from_numpy(gt_clean)
        
        return {'id': fissure_name, 'visible': vis_t, 'infrared': ir_t, 'gt': gt_t}

raphael_dataset = RaphaelDatasetSubset('.')



In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Same default parameters as CrackForest (single modality => visible only)
# However, for Raphael, we use the fusion of both modalities.
weights_raphael = {'visible': 0.5, 'infrared': 0.5}

num_eval_raphael = len(raphael_dataset)
results_raphael = []

print(f"Evaluation on the Raphael dataset ({num_eval_raphael} images)...")
import matplotlib.pyplot as plt

for i in range(num_eval_raphael):
    sample_i = raphael_dataset[i]
    imgs_i = {'visible': sample_i['visible'], 'infrared': sample_i['infrared']}
    
    # Function call with default parameters + K=2 and Σ=[20.0] for the benchmark
    frangi_response, similarity_img, centrality_i, timings, diagnostics = extract_frangi_graph_gpu(imgs_i, weights_raphael, Σ=[20.0], K=2, device=device)
    
    pred_i = (centrality_i > 0.025).astype(np.uint8)
    pred_i = skeletonize_lee(pred_i)
    sk_pred_thick_i = thicken(pred_i, pixels=3)
    
    gt_arr_i = sample_i['gt'].numpy().astype(np.uint8)
    sk_gt_i = skeletonize_lee(gt_arr_i)
    sk_gt_thick_i = thicken(sk_gt_i, pixels=3)
    
    j, t = compute_metrics(sk_pred_thick_i, sk_gt_thick_i)
    w = wasserstein_distance_skeletons(sk_pred_thick_i, sk_gt_thick_i)
    
    results_raphael.append({
        'ID': sample_i['id'],
        'Jaccard (IoU)': j,
        'Tversky': t,
        'Wasserstein': w
    })
    
    print(f"--- Metrics for sample: {sample_i['id']} ---")
    print(f"Jaccard (IoU): {j:.4f}")
    print(f"Tversky:       {t:.4f}")
    print(f"Wasserstein:   {w:.4f}")
    
    fig, axes = plt.subplots(2, 5, figsize=(40, 12))
    
    axes[0, 0].imshow(sample_i['visible'].numpy(), cmap='gray')
    axes[0, 0].set_title('Modality: Visible')

    axes[0, 1].imshow(sample_i['infrared'].numpy(), cmap='gray')
    axes[0, 1].set_title('Modality: Thermal')
    
    axes[0, 2].imshow(frangi_response, cmap='magma')
    axes[0, 2].set_title('Frangi Response (Fused Λ2)')
    
    axes[0, 3].imshow(similarity_img, cmap='magma')
    axes[0, 3].set_title('Frangi-Graph Similarity (Max)')
    
    axes[1, 1].imshow(sample_i['gt'].numpy(), cmap='gray')
    axes[1, 1].set_title('Ground Truth')
    
    axes[1, 0].imshow(centrality_i, cmap='hot')
    axes[1, 0].set_title('Betweenness Centrality')
    
    skeleton = pred_i
    axes[0, 4].imshow(np.zeros_like(skeleton), cmap='gray')
    h, w_dim = skeleton.shape
    rgba_tau = np.zeros((h, w_dim, 4), dtype=np.float32)
    rgba_tau[diagnostics['tau_mask'] > 0] = [1.0, 1.0, 1.0, 0.3]
    rgba_comp = np.zeros((h, w_dim, 4), dtype=np.float32)
    rgba_comp[diagnostics['comp_mask'] > 0] = [0.0, 0.5, 1.0, 0.8]
    axes[0, 4].imshow(rgba_tau)
    axes[0, 4].imshow(rgba_comp)
    axes[0, 4].set_title('Selected Connected Components')
    
    axes[1, 2].imshow(skeleton, cmap='gray')
    axes[1, 2].set_title('Predicted Skeleton (Raw)')
    
    axes[1, 3].imshow(np.zeros_like(skeleton), cmap='gray')
    rgba_gt_skel = np.zeros((h, w_dim, 4), dtype=np.float32)
    rgba_gt_skel[sk_gt_thick_i > 0] = [1.0, 0.0, 0.0, 0.4] # Transparent red
    
    rgba_pred = np.zeros((h, w_dim, 4), dtype=np.float32)
    rgba_pred[sk_pred_thick_i > 0] = [0.0, 1.0, 0.0, 0.4] # Transparent green
    
    axes[1, 3].imshow(rgba_gt_skel)
    axes[1, 3].imshow(rgba_pred)
    axes[1, 3].set_title('Eval (Red: GT, Green: Pred)')

    axes[1, 4].axis('off')
    
    for ax in axes.flat:
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

if results_raphael:
    df_results_raphael = pd.DataFrame(results_raphael)
    display(df_results_raphael)

    print("\n--- Global Statistics (Raphael Cracks 1-3) ---")
    print(f"Average Jaccard (IoU): {df_results_raphael['Jaccard (IoU)'].mean():.4f} ± {df_results_raphael['Jaccard (IoU)'].std():.4f}")
    print(f"Average Tversky:        {df_results_raphael['Tversky'].mean():.4f} ± {df_results_raphael['Tversky'].std():.4f}")
    print(f"Average Wasserstein:    {df_results_raphael['Wasserstein'].mean():.4f} ± {df_results_raphael['Wasserstein'].std():.4f}")
else:
    print("No results to display (dataset not found or empty).")



## 5. Parameter Sensitivity Analysis

We will vary parameters one by one: `largeur_Sigma` (width of the scale set), `R, ss, si, sa, τ`, `τ_c` (centrality threshold), `min_rel_size` (minimum relative component size) et `σ_0` (central scale).
Other parameters will remain constant.

We first load the dataset into RAM for ultra-fast execution.


In [ ]:
import copy
from tqdm.auto import tqdm
import pandas as pd

print("Loading all 118 images into RAM to speed up sensitivity analysis...")
all_data = []
for i in range(len(dataset)):
    all_data.append(dataset[i])
print("Done.")

default_params = {
    'K': 1,
    'largeur_Sigma': 0,
    'R': 5,
    'ss': 1.0,
    'si': 0.25,
    'sa': 0.3,
    'τ': 0.2,
    'σ_0': 5.0,
    'τ_c': 0.025,
    'min_rel_size': 200.0
}

# nb_pas = 10
param_ranges = {
    'K': [1, 2],
    'largeur_Sigma': [0, 1, 2],
    'R': np.linspace(1, 10, 10, dtype=int).tolist(),
    'ss': np.linspace(0.5, 2.5, 10).tolist(),
    'si': np.linspace(0.1, 0.9, 10).tolist(),
    'sa': np.linspace(0.1, 0.8, 10).tolist(),
    'τ': np.linspace(0.05, 0.4, 10).tolist(),
    'τ_c': np.linspace(0.01, 0.06, 10).tolist(),
    'σ_0': np.linspace(2.0, 10.0, 10).tolist(),
    'min_rel_size': np.linspace(50.0, 400.0, 10).tolist()
}

os.makedirs("sensitivity_results", exist_ok=True)

def evaluate_dataset(params):
    import time
    j_list, t_list, w_list = [], [], []
    individual_results = []
    
    sigma_val = params['σ_0']
    largeur = int(params['largeur_Sigma'])
    Sigma_list = [sigma_val + i for i in range(-largeur, largeur + 1)]
    k_val = params.get('K', 1)
    
    for sample in tqdm(all_data, desc=f"Eval images (K={k_val}, Σ={Sigma_list}, R={params['R']}, ss={params['ss']:.2f}, si={params['si']:.2f}, sa={params['sa']:.2f}, τ={params['τ']:.2f}, τ_c={params['τ_c']:.3f}, min_rel_size={params['min_rel_size']:.1f})", leave=False):
        imgs_i = {'visible': sample['visible']}
        weights_i = {'visible': 1.0}
        
        _, _, centrality_i, _, _ = extract_frangi_graph_gpu(
            imgs_i, weights_i, 
            Σ=Sigma_list, 
            R=int(params['R']),
            ss=params['ss'], 
            si=params['si'], 
            sa=params['sa'], 
            τ=params['τ'],
            min_rel_size=params['min_rel_size'],
            K=k_val,
            device=device
        )
        
        pred_i = (centrality_i > params['τ_c']).astype(np.uint8)
        if k_val == 2:
            pred_i = skeletonize_lee(pred_i)
        sk_pred_thick_i = thicken(pred_i, pixels=3)
        
        gt_arr_i = sample['gt'].numpy().astype(np.uint8)
        sk_gt_i = skeletonize_lee(gt_arr_i)
        sk_gt_thick_i = thicken(sk_gt_i, pixels=3)
        
        j, t = compute_metrics(sk_pred_thick_i, sk_gt_thick_i)
        w = wasserstein_distance_skeletons(sk_pred_thick_i, sk_gt_thick_i)
        
        j_list.append(j)
        t_list.append(t)
        w_list.append(w)
        individual_results.append({
            'ID': sample['id'], 
            'Jaccard': j, 
            'Tversky': t, 
            'Wasserstein': w
        })
        
    return np.mean(j_list), np.std(j_list), np.mean(t_list), np.std(t_list), np.mean(w_list), np.std(w_list), individual_results

import time
for param_name, values in param_ranges.items():
    print(f"\n{'='*60}\n--- Sensitivity analysis for {param_name} ---\n{'='*60}")
    results_summary = []
    all_individual_results = []
    
    t0_param = time.time()
    
    for val in values:
        t0_val = time.time()
        
        current_params = copy.deepcopy(default_params)
        current_params[param_name] = val
        
        j_mean, j_std, t_mean, t_std, w_mean, w_std, ind_res = evaluate_dataset(current_params)
        
        t_val = time.time() - t0_val
        print(f"[{param_name} = {val}] evaluated in {t_val:.2f}s : Jaccard={j_mean:.4f}±{j_std:.4f} | Tversky={t_mean:.4f}±{t_std:.4f} | Wasserstein={w_mean:.4f}±{w_std:.4f}")
        
        results_summary.append({
            param_name: val,
            'Jaccard_mean': j_mean, 'Jaccard_std': j_std,
            'Tversky_mean': t_mean, 'Tversky_std': t_std,
            'Wasserstein_mean': w_mean, 'Wasserstein_std': w_std
        })
        
        for res in ind_res:
            res_copy = res.copy()
            res_copy[param_name] = val
            all_individual_results.append(res_copy)
            
    t_param = time.time() - t0_param
    print(f"--- End of analysis for {param_name} (Total time: {t_param:.2f}s) ---")
            
    # CSV Save
    df_summary = pd.DataFrame(results_summary)
    df_summary.to_csv(f"sensitivity_results/summary_{param_name}.csv", index=False)
    
    df_ind = pd.DataFrame(all_individual_results)
    df_ind.to_csv(f"sensitivity_results/individual_{param_name}.csv", index=False)
    
    # Display the 3 resulting curves
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Sensitivity: {param_name}", fontsize=16)
    
    x_vals = df_summary[param_name]
    
    # Jaccard
    axes[0].plot(x_vals, df_summary['Jaccard_mean'], 'b-', label='Jaccard')
    axes[0].fill_between(x_vals, 
                         df_summary['Jaccard_mean'] - df_summary['Jaccard_std'],
                         df_summary['Jaccard_mean'] + df_summary['Jaccard_std'], 
                         color='b', alpha=0.2)
    axes[0].set_title('Jaccard (IoU)')
    axes[0].set_xlabel(param_name)
    axes[0].set_ylabel('Score')
    axes[0].set_ylim(0.30, 0.80)
    axes[0].grid(True)
    
    # Tversky
    axes[1].plot(x_vals, df_summary['Tversky_mean'], 'g-', label='Tversky')
    axes[1].fill_between(x_vals, 
                         df_summary['Tversky_mean'] - df_summary['Tversky_std'],
                         df_summary['Tversky_mean'] + df_summary['Tversky_std'], 
                         color='g', alpha=0.2)
    axes[1].set_title('Tversky')
    axes[1].set_xlabel(param_name)
    axes[1].set_ylabel('Score')
    axes[1].set_ylim(0.30, 0.80)
    axes[1].grid(True)
    
    # Wasserstein
    axes[2].plot(x_vals, df_summary['Wasserstein_mean'], 'r-', label='Wasserstein')
    axes[2].fill_between(x_vals, 
                         df_summary['Wasserstein_mean'] - df_summary['Wasserstein_std'],
                         df_summary['Wasserstein_mean'] + df_summary['Wasserstein_std'], 
                         color='r', alpha=0.2)
    axes[2].set_title('Wasserstein')
    axes[2].set_xlabel(param_name)
    axes[2].set_ylabel('Distance')
    axes[2].set_ylim(0, 100)
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.savefig(f"sensitivity_results/plot_{param_name}.png")
    plt.show()


## 6. Interaction Analysis Between R and $\sigma_0$ (2D Grid Search)

We will perform a Grid Search on parameters `R` et `\sigma_0` to verify whether they are independent or interact.
We will test a 5x5 grid (25 dataset evaluations) and visualize the results as Heatmaps.
Finally, we will perform a **Two-Way ANOVA** to statistically test the interaction.

The underlying mathematical model is formally written as:
$$Y_{ijk} = \mu + \alpha_i + \beta_j + (\alpha\beta)_{ij} + \epsilon_{ijk}$$

Where:
- $Y_{ijk}$ is the $k$-th observation of the result for level $i$ of parameter $R$ and level $j$ of parameter $\sigma_0$.
- $\mu$ is the theoretical grand mean.
- $\alpha_i$ is the main effect of level $i$ of parameter $R$.
- $\beta_j$ is the main effect of level $j$ of parameter $\sigma_0$.
- $(\alpha\beta)_{ij}$ is the interaction effect between level $i$ of parameter $R$ and level $j$ of parameter $\sigma_0$.
- $\epsilon_{ijk}$ is the residual error, which must follow a normal distribution $\mathcal{N}(0, \sigma^2)$.

### Testing "Result Independence": Interaction Effect and Null Hypothesis $H_0$

The null hypothesis $H_0$ postulates the absence of interaction: $(\alpha\beta)_{ij} = 0 \ \forall \ i,j$.
To rigorously demonstrate the properties of these estimators under $H_0$, a balanced experimental design is established. It is not the Mean Squares ($MS$) that follow a $\chi^2$ distribution, but the **Sums of Squares ($SS$) normalized by the theoretical population variance $\sigma^2$**. 

1. **Residual Sum of Squares ($SS_{residual}$)**
It quantifies the dispersion of raw observations around the mean of their own cell.
$$SC_{résiduel} = \sum_{i=1}^{a} \sum_{j=1}^{b} \sum_{k=1}^{n} (Y_{ijk} - \bar{Y}_{ij.})^2$$
Knowing that the expected deviation is zero, Cochran's theorem proves that the sum of these normalized squares follows a $\chi^2$ distribution (whether $H_0$ is true or false):
$$\frac{SC_{résiduel}}{\sigma^2} \sim \chi^2(ab(n-1))$$

2. **Interaction Sum of Squares ($SS_{AB}$)**
It quantifies the deviation between a cell's observed mean and what a purely additive model based on marginal effects would predict.
$$SC_{AB} = n \sum_{i=1}^{a} \sum_{j=1}^{b} (\bar{Y}_{ij.} - \bar{Y}_{i..} - \bar{Y}_{.j.} + \bar{Y}_{...})^2$$
**If and only if** the null hypothesis of strict additivity ($H_0$) is true, the expected value of the interaction estimator is zero. Cochran's theorem then proves that this quadratic form follows a $\chi^2$ distribution:
$$\frac{SC_{AB}}{\sigma^2} \sim \chi^2((a-1)(b-1))$$

3. **Consequence on the F-statistic**
The ultimate goal is to eliminate the unknown theoretical variance $\sigma^2$ by calculating the ratio of two independent $\chi^2$ variables, each divided by its degrees of freedom ($df$). This corresponds to the Fisher distribution:
$$F_{AB} = \frac{ \left( \frac{SC_{AB}}{\sigma^2} \right) / ddl_{AB} }{ \left( \frac{SC_{résiduel}}{\sigma^2} \right) / ddl_{résiduel} } = \frac{ \left( \frac{SC_{AB}}{ddl_{AB}} \right) }{ \left( \frac{SC_{résiduel}}{ddl_{résiduel}} \right) } = \frac{CM_{AB}}{CM_{résiduel}}$$
The $F_{AB}$ statistic is calculable exclusively from empirical data, and its critical probability (p-value) can be evaluated knowing that it follows, under $H_0$, an distribution $\mathcal{F}((a-1)(b-1), ab(n-1))$.



In [ ]:
!pip install -q statsmodels seaborn

import time
import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import ols
import itertools
import numpy as np
import matplotlib.pyplot as plt

print("\n" + "="*60 + "\n--- 2D Grid Search: R vs σ_0 ---\n" + "="*60)

R_values = np.linspace(2, 10, 5, dtype=int).tolist()
sigma_values = np.linspace(2.0, 10.0, 5).tolist()

grid_results_summary = []
grid_individual_results = []

t0_grid = time.time()

# For the grid, we fix largeur_Sigma to 0 (only the central scale)
# because sigma_0 can vary, which would change the number of scales if largeur_Sigma > 0
for R, sigma in tqdm(list(itertools.product(R_values, sigma_values)), desc="Grid Search R x σ_0"):
    current_params = copy.deepcopy(default_params)
    current_params['R'] = R
    current_params['σ_0'] = sigma
    current_params['largeur_Sigma'] = 0  # Fixé à 0 pour l'étude
    
    j_mean, j_std, t_mean, t_std, w_mean, w_std, ind_res = evaluate_dataset(current_params)
    
    grid_results_summary.append({
        'R': R,
        'sigma_0': sigma,
        'Jaccard': j_mean,
        'Tversky': t_mean,
        'Wasserstein': w_mean
    })
    
    for res in ind_res:
        res_copy = res.copy()
        res_copy['R'] = R
        res_copy['sigma_0'] = sigma
        grid_individual_results.append(res_copy)

t_grid = time.time() - t0_grid
print(f"--- End of Grid Search (Total time: {t_grid:.2f}s) ---")

df_grid_summary = pd.DataFrame(grid_results_summary)
df_grid_ind = pd.DataFrame(grid_individual_results)

# --- 1. Heatmaps ---
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle(r"Heatmaps: Interaction between R and $\sigma_0$", fontsize=16)

# We round indices/columns for clean display
df_grid_summary['sigma_0_round'] = df_grid_summary['sigma_0'].round(1)
df_grid_summary['R_round'] = df_grid_summary['R'].astype(int)

pivot_jaccard = df_grid_summary.pivot(index='sigma_0_round', columns='R_round', values='Jaccard')
pivot_tversky = df_grid_summary.pivot(index='sigma_0_round', columns='R_round', values='Tversky')
pivot_wasserstein = df_grid_summary.pivot(index='sigma_0_round', columns='R_round', values='Wasserstein')

sns.heatmap(pivot_jaccard, ax=axes[0], cmap='viridis', annot=True, fmt=".3f", vmin=0, vmax=1, cbar_kws={'label': 'Jaccard'})
axes[0].set_title('Jaccard (IoU)')
axes[0].invert_yaxis()

sns.heatmap(pivot_tversky, ax=axes[1], cmap='viridis', annot=True, fmt=".3f", vmin=0, vmax=1, cbar_kws={'label': 'Tversky'})
axes[1].set_title('Tversky')
axes[1].invert_yaxis()

sns.heatmap(pivot_wasserstein, ax=axes[2], cmap='plasma_r', annot=True, fmt=".1f", vmin=0, vmax=150, cbar_kws={'label': 'Wasserstein'})
axes[2].set_title('Wasserstein')
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig("sensitivity_results/grid_search_heatmaps.png")
plt.show()

# --- 2. ANOVA à deux facteurs ---
print("\n--- Two-Way ANOVA ---")
# We rename 'sigma_0' to 'sigma' to avoid issues with patsy formulas if needed
df_grid_ind_anova = df_grid_ind.rename(columns={'sigma_0': 'sigma'})

def print_anova_results(model, metric_name):
    anova_table = sm.stats.anova_lm(model, typ=2)
    print(f"\nANOVA - Dependent Variable: {metric_name}")
    print(anova_table)
    
    interaction_term = 'C(R):C(sigma)'
    if interaction_term in anova_table.index:
        p_value = anova_table.loc[interaction_term, 'PR(>F)']
        f_stat = anova_table.loc[interaction_term, 'F']
        
        print(f"\n>>> H_0 HYPOTHESIS TEST (Independence of R and sigma_0) on {metric_name} <<<")
        print(f"Fisher Statistic F_AB = {f_stat:.4f}")
        print(f"Critical probability (p-value) = {p_value:.4e}")
        
        alpha = 0.05
        if p_value < alpha:
            print(f"Conclusion: p-value < {alpha} (5% Significance level).")
            print("             => We firmly REJECT the null hypothesis H_0.")
            print("             => The interaction effect is statistically significant.")
        else:
            print(f"Conclusion: p-value >= {alpha} (5% Significance level).")
            print("             => We ACCEPT (or cannot reject) the null hypothesis H_0.")
            print("             => The parameters appear to act independently.")

# Model for Jaccard
model_jaccard = ols('Jaccard ~ C(R) + C(sigma) + C(R):C(sigma)', data=df_grid_ind_anova).fit()
print_anova_results(model_jaccard, "Jaccard")

# Model for Tversky
model_tversky = ols('Tversky ~ C(R) + C(sigma) + C(R):C(sigma)', data=df_grid_ind_anova).fit()
print_anova_results(model_tversky, "Tversky")

